# Image → 3D Scene (Stage B · TripoSR)

Trellis can't install on Colab's Python 3.12 (`spconv` has no wheel). This notebook uses **TripoSR** instead — fast, installs clean, no custom CUDA ops — to generate a real 3D mesh per detected object, placed on a fitted ground plane → `scene.glb`.

**Before running:** `Runtime → Change runtime type → T4 GPU`. Run cells top to bottom.

In [ ]:
# 0. Confirm GPU
!nvidia-smi -L

## 1. Install TripoSR + deps (~3–5 min)

In [ ]:
import sys
![ -d /content/TripoSR ] || git clone https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR
%cd /content/TripoSR
# TripoSR's own deps (includes torchmcubes, omegaconf, einops, rembg, …)
!pip install -q -r requirements.txt
# Our detection / depth / geometry deps (install AFTER so transformers isn't downgraded)
!pip install -q "transformers>=4.44,<5" timm accelerate huggingface_hub trimesh xatlas scipy
if '/content/TripoSR' not in sys.path:
    sys.path.insert(0, '/content/TripoSR')
# Sanity: TripoSR must import
from tsr.system import TSR  # noqa: F401
print('OK: TripoSR imports')

## 2. Get the pipeline code

In [ ]:
REPO = '/content/image-3d-pipeline'
BRANCH = 'claude/sweet-cori-kzkhyr'
![ -d {REPO} ] || git clone --branch {BRANCH} https://github.com/sanjanamani/image-3d-pipeline.git {REPO}
!cd {REPO} && git fetch origin {BRANCH} -q && git checkout {BRANCH} -q && git pull -q
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('pipeline code ready:', REPO)

## 3. Upload your photo

In [ ]:
from google.colab import files
uploaded = files.upload()
IMAGE_PATH = '/content/' + next(iter(uploaded))
print('uploaded:', IMAGE_PATH)

## 4. Build the 3D scene (TripoSR per object)

In [ ]:
import os
os.chdir(REPO)
from scene_build import run_build
glb_path = run_build(IMAGE_PATH, output_dir='/content/outputs', backend='triposr')
print('\nDONE ->', glb_path)

## 5. View inline + download

In [ ]:
import base64
from IPython.display import HTML, display

b64 = base64.b64encode(open(glb_path, 'rb').read()).decode()
display(HTML(f'''
<script type="module" src="https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js"></script>
<model-viewer src="data:model/gltf-binary;base64,{b64}"
  camera-controls auto-rotate shadow-intensity="1"
  style="width:100%;height:520px;background:#222;"></model-viewer>
'''))

from google.colab import files as _f
_f.download(glb_path)